# B1.2 · Component summarisation and architecture synthesis

**Function B — Product & Application Security → The AppSec Engineer / Code Reviewer**  ·  *AI for Security*

Builds on **[B1.1 · Historical parsing and structural indexing](https://spbreed.github.io/cyber-commons/lessons/B1.1.html)**.

| | |
|---|---|
| Open-source tooling | tree-sitter, Graphviz |
| Open-weight models | GLM-4.6, Kimi K2 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Stages 1 and 2 produced units and their call relationships. That is still a
pile of functions. Phase 1 finishes by turning it into something a threat model
can be derived from.

**Stage 3 — Component summarisation.** Generate a localised summary per
directory or module: what it is for, what it talks to, what data passes through
it. Localised is the important word — summarising the whole repository at once
produces a paragraph that is true of every repository.

**Stage 4 — Architecture synthesis.** Compile those summaries into a single map
with three things on it:

- **entry points** — where untrusted input arrives,
- **data flows** — how it travels between components,
- **trust boundaries** — where it crosses from less trusted to more trusted.

The map is the artefact. Every later stage consumes it: threat modelling reads
the boundaries, planning allocates against them, feasibility filtering walks the
flows.

> **Where you are in the pipeline.**
>
> ```
> [Ingestion & Mapping] ──> [Threat Modelling] ──> [Discovery]
>          └─ stages 1-4         └─ stages 5-6        └─ stages 7-10
>                    ──> [Dynamic Validation] ──> [Reporting]
>                              └─ stages 11-14        └─ stage 15
> ```

> **About the model in this notebook.** It runs offline against a deterministic
> stand-in so the lesson executes on a Kaggle kernel with no network. The
> stand-in is not a language model and is labelled as such wherever it appears.
> To run the identical pipeline stage against a real open-weight model:
>
> ```bash
> ollama pull glm-4.6            # or kimi-k2, llama3.3
> export OPENAI_BASE_URL=http://localhost:11434/v1 OPENAI_API_KEY=ollama MODEL=glm-4.6
> ```

## 2 · Stage 3 — summarise each component, locally

In [ ]:
import ast
from dataclasses import dataclass, field
from collections import defaultdict

SOURCES = {
 "src/web/handlers.py": '''
def get_report(request):
    """HTTP GET /reports/<id> — request.args is user-controlled."""
    return render(load_report(request.args["id"], request.args["owner"]))

def upload_doc(request):
    """HTTP POST /docs — multipart body is user-controlled."""
    return store(request.files["doc"], request.args["name"])
''',
 "src/data/reports.py": '''
def load_report(report_id, owner):
    return DB.execute("SELECT * FROM reports WHERE id=" + report_id +
                      " AND owner='" + owner + "'")
''',
 "src/data/docs.py": '''
def store(blob, name):
    path = "/srv/docs/" + name
    open(path, "wb").write(blob)
    return path
''',
 "src/util/render.py": '''
def render(rows):
    return "\\n".join(str(r) for r in rows)
''',
}

def units_of(src, path):
    tree = ast.parse(src)
    out = []
    for fn in [n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)]:
        calls = sorted({(c.func.id if isinstance(c.func, ast.Name)
                         else getattr(c.func, "attr", ""))
                        for c in ast.walk(fn) if isinstance(c, ast.Call)} - {""})
        doc = ast.get_docstring(fn) or ""
        # file + line, never a bare name: two `handler` functions in different
        # files are two different units, and merging them loses a finding
        out.append({"name": fn.name, "file": path, "line": fn.lineno,
                    "params": [a.arg for a in fn.args.args],
                    "calls": calls, "doc": doc})
    return out

ALL_UNITS = [u for p, s in SOURCES.items() for u in units_of(s, p)]

DANGEROUS = {"execute": "database", "open": "filesystem", "write": "filesystem",
             "system": "shell", "get": "network"}

def summarise_component(directory, units):
    """Stage 3 — a LOCAL summary. Deterministic here; a model does this in production."""
    names = [u["name"] for u in units]
    external = sorted({c for u in units for c in u["calls"]
                       if c not in names and c in DANGEROUS})
    outbound = sorted({c for u in units for c in u["calls"] if c in
                       {x["name"] for x in ALL_UNITS} and c not in names})
    entry = [u["name"] for u in units if u["doc"].startswith("HTTP")]
    return {"component": directory, "units": names, "entry_points": entry,
            "talks_to": outbound,
            "touches": sorted({DANGEROUS[c] for c in external})}

by_dir = defaultdict(list)
for u in ALL_UNITS:
    by_dir["/".join(u["file"].split("/")[:-1])].append(u)

SUMMARIES = [summarise_component(d, us) for d, us in sorted(by_dir.items())]
for s in SUMMARIES:
    print(f"{s['component']}")
    print(f"   units       {s['units']}")
    print(f"   entry pts   {s['entry_points'] or '—'}")
    print(f"   talks to    {s['talks_to'] or '—'}")
    print(f"   touches     {s['touches'] or '—'}")
    print()

## 3 · Stage 4 — synthesise the architecture map

In [ ]:
def synthesise(summaries, units):
    by_name = {u["name"]: u for u in units}
    entry_points, flows, sinks = [], [], []
    for s in summaries:
        for e in s["entry_points"]:
            entry_points.append({"unit": e, "component": s["component"],
                                 "input": "HTTP request (untrusted)"})
    for u in units:
        for c in u["calls"]:
            if c in by_name:
                flows.append((u["name"], c))
            elif c in DANGEROUS:
                sinks.append({"unit": u["name"], "sink": c,
                              "resource": DANGEROUS[c]})
    return {"entry_points": entry_points, "flows": sorted(set(flows)), "sinks": sinks}

MAP = synthesise(SUMMARIES, ALL_UNITS)
print("ENTRY POINTS (untrusted input arrives here)")
for e in MAP["entry_points"]:
    print(f"   {e['unit']:14s} {e['component']:22s} {e['input']}")
print("\nDATA FLOWS")
for a, b in MAP["flows"]:
    print(f"   {a} → {b}")
print("\nSINKS (state changes / external resources)")
for s in MAP["sinks"]:
    print(f"   {s['unit']:14s} {s['sink']:10s} {s['resource']}")

## 4 · Trust boundaries — the part the map exists for

A boundary is any edge where data crosses from a less-trusted component into a more-trusted one. Those edges are where every finding in the rest of the pipeline will turn out to live.

In [ ]:
TRUST = {"src/web": 0, "src/data": 2, "src/util": 1}   # 0 = untrusted edge

def boundaries(flows, units):
    comp = {u["name"]: "/".join(u["file"].split("/")[:-1]) for u in units}
    out = []
    for a, b in flows:
        ca, cb = comp[a], comp[b]
        if TRUST.get(ca, 0) < TRUST.get(cb, 0):
            out.append({"edge": f"{a} → {b}", "from": ca, "to": cb,
                        "crossing": f"trust {TRUST[ca]} → {TRUST[cb]}"})
    return out

B = boundaries(MAP["flows"], ALL_UNITS)
print("TRUST BOUNDARY CROSSINGS")
for b in B:
    print(f"   {b['edge']:28s}{b['from']:10s} → {b['to']:10s} ({b['crossing']})")

reachable_sinks = []
entry_names = {e["unit"] for e in MAP["entry_points"]}
adj = defaultdict(list)
for a, b in MAP["flows"]: adj[a].append(b)
def walk(start, seen=None):
    seen = seen or set()
    if start in seen: return set()
    seen |= {start}
    out = {start}
    for n in adj[start]: out |= walk(n, seen)
    return out
# sorted(), not the set itself: a reachability report that lists the same
# entry points in a different order on every machine cannot be diffed between
# two scans, and diffing scans is the whole point of mapping the architecture.
for e in sorted(entry_names):
    for s in MAP["sinks"]:
        if s["unit"] in walk(e):
            reachable_sinks.append((e, s["unit"], s["resource"]))
print("\nSINKS REACHABLE FROM AN ENTRY POINT")
for e, u, res in reachable_sinks:
    print(f"   {e:14s} → {u:14s} touches {res}")
assert reachable_sinks

In [ ]:
# Verify: the map must change when the architecture changes.
SOURCES_V2 = dict(SOURCES)
SOURCES_V2["src/web/handlers.py"] = SOURCES["src/web/handlers.py"] + '''
def admin_export(request):
    """HTTP GET /admin/export — user-controlled, previously internal only."""
    return store(load_report(request.args["id"], request.args["owner"]),
                 request.args["name"])
'''
units_v2 = [u for p, s in SOURCES_V2.items() for u in units_of(s, p)]
by_dir2 = defaultdict(list)
for u in units_v2: by_dir2["/".join(u["file"].split("/")[:-1])].append(u)
map_v2 = synthesise([summarise_component(d, us) for d, us in sorted(by_dir2.items())],
                    units_v2)

before = {e["unit"] for e in MAP["entry_points"]}
after  = {e["unit"] for e in map_v2["entry_points"]}
print(f"entry points before: {sorted(before)}")
print(f"entry points after : {sorted(after)}")
print(f"NEW ENTRY POINT    : {sorted(after - before)}")
print(f"flows before {len(MAP['flows'])} → after {len(map_v2['flows'])}")
print("\nOne function added. A new untrusted entry point now reaches both the")
print("database and the filesystem. That delta is what B1.3 threat-models.")
assert after - before

## 6 · Write the procedure down as an agent skill

You have just run four stages by hand. The next repository needs the same four,
and so does the next agent. An **agent skill** is how that procedure stops
living in your head.

A skill is a markdown file with a small header:

```
---
name: appsec-repo-recon
description: >-
  Build the structural and historical map of a codebase before any security
  analysis. Use at the start of an application security review, when asked to
  find entry points, sinks, trust boundaries or attack surface ...
allowed-tools: Read, Grep, Glob, Bash
---

# the procedure, written for whoever runs it next
```

Three fields, three different jobs:

- **`name`** identifies it.
- **`description`** is the **routing key**, not documentation. An agent decides
  whether to load a skill by reading this sentence and nothing else. A
  description that says "helps with security stuff" never fires, and two
  descriptions that overlap fire the wrong one.
- **`allowed-tools`** bounds it. This skill reads a repository; it never writes
  to one, and that is enforceable rather than merely stated.

The body carries the procedure and — the part that matters here — an **output
contract**: the exact JSON shape Phase 2 will join against. A skill with a
contract is testable. A skill without one is a wish.

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

In [ ]:
# skills/appsec/appsec-repo-recon/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: appsec-repo-recon
description: >-
  Build the structural and historical map of a codebase before any security
  analysis. Use at the start of an application security review, when asked to
  find entry points, sinks, trust boundaries or attack surface, when triaging
  which files deserve attention, or when a later stage needs an architecture
  map it does not have.
allowed-tools: Read, Grep, Glob, Bash
---

# AppSec pipeline · Phase 1 — Ingestion and structural mapping

Covers **stages 1–4**. Produces the map every later stage reads. Nothing in
this phase decides whether the code is vulnerable — it decides *where to look*.

Most review starts at the diff, which is the smallest context available and
discards the best predictor there is: the repository has already recorded where
it breaks.

## When to use this

Load this skill first in any review of a codebase you have not mapped. If a
threat model, audit or report is requested and no `architecture_map` exists,
build one here rather than guessing from file names.

## Inputs

| Input | Required | Notes |
|---|---|---|
| Repository worktree | yes | read-only is sufficient |
| Commit history | preferred | degraded mode without it; say so in `caveats` |
| Issue/PR history | optional | improves stage 1 only |

## Procedure

**Stage 1 — Historical parsing.** Extract prior vulnerabilities, the commits
that fixed them, and their files. Security fixes cluster: a file patched for a
vulnerability once is materially more likely to hold another. Record a
`fix_count` per file. Never treat absence of history as evidence of safety —
it is usually evidence of a young file or a squashed import.

**Stage 2 — Structural indexing.** Enumerate units (functions, methods,
handlers) with file, line, parameters, and the calls each one makes. This is
the index every later stage joins against, so record identity as
`(file, unit)` — never a bare basename. Two `handler.py` files in different
directories are two different units, and collapsing them silently merges their
findings.

**Stage 3 — Component summarisation.** For each unit, record what it *touches*:
network, filesystem, database, subprocess, credentials, deserialisation. A unit
that touches none of these cannot be a sink, and excluding it early is the
cheapest correct filter in the pipeline.

**Stage 4 — Architecture synthesis.** Join the above into:
- **entry points** — units reachable from outside the trust boundary
- **sinks** — units that touch a dangerous resource
- **flows** — the call edges connecting them
- **trust boundaries** — the edges where the caller's trust level drops

Then compute reachability: for every entry point, which sinks can it reach.
An unreachable sink is not an attack surface, and a reachable one is the whole
list for Phase 2.

## Output contract

Emit exactly this shape. Later phases join on these keys.

```json
{
  "architecture_map": {
    "entry_points": [{"unit": "str", "file": "str", "line": 0, "exposure": "public|authenticated|internal"}],
    "sinks":        [{"unit": "str", "file": "str", "resource": "network|filesystem|database|subprocess|credential|deserialisation"}],
    "flows":        [["caller_unit", "callee_unit"]],
    "boundaries":   [{"edge": "a → b", "from_trust": 0, "to_trust": 0}],
    "reachable":    [{"entry": "str", "sink": "str", "path": ["unit", "..."]}],
    "hotspots":     [{"file": "str", "fix_count": 0}],
    "caveats":      ["str"]
  }
}
```

Order every list deterministically — sort by a full key, never rely on set or
dict iteration order. Two runs of this skill over the same commit must produce
byte-identical output, or the diff between two scans is meaningless.

## Failure modes

- **Reporting a sink with no path from an entry point.** That is a code smell,
  not an attack surface. Keep it out of `reachable`.
- **Matching paths by basename.** Match on parent directory plus filename tail.
- **Claiming completeness.** If the index skipped a language, generated code,
  or a vendored tree, list it in `caveats`. A silently partial map is worse
  than a small one, because the next stage cannot tell.

## Handoff

Pass `architecture_map` to **appsec-threat-model**. If `reachable` is empty,
stop and report that — do not proceed to threat modelling on an empty surface.
"""

meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

## 7 · The contract is executable — check the map you just built

The skill promised a shape. You built a map. Those two claims can be checked against each other mechanically, which is the whole reason to write the contract down.

In [ ]:
# Express the map this lesson built in the shape the skill promises.
contract = contract_of(body)
at = {u["name"]: u for u in ALL_UNITS}
EXPOSURE = {"src/web": "public", "src/util": "internal", "src/data": "internal"}

recon = {"architecture_map": {
  "entry_points": [
      {"unit": e["unit"], "file": at[e["unit"]]["file"], "line": at[e["unit"]]["line"],
       "exposure": EXPOSURE.get("/".join(at[e["unit"]]["file"].split("/")[:-1]), "internal")}
      for e in MAP["entry_points"]],
  "sinks": [
      {"unit": s["unit"], "file": at[s["unit"]]["file"], "resource": s["resource"]}
      for s in MAP["sinks"]],
  "flows": [[a, b] for a, b in MAP["flows"]],
  "boundaries": [
      {"edge": b["edge"], "from_trust": TRUST[b["from"]], "to_trust": TRUST[b["to"]]}
      for b in B],
  "reachable": [
      {"entry": e, "sink": u, "path": [e, u]} for e, u, _ in reachable_sinks],
  # carried forward from stage 1 in B1.1 — each notebook stands alone, so the
  # result of the previous stage arrives as a literal rather than an import
  "hotspots": [{"file": "src/auth.py", "fix_count": 3},
               {"file": "src/data/reports.py", "fix_count": 1}],
  "caveats": ["single language; vendored trees not indexed"],
}}

problems = check(recon, contract)
print(f"conformance check: {len(problems)} problem(s)")
for p in problems:
    print("   ", p)
assert not problems, problems
print("\nThe map satisfies the contract, so Phase 2 can consume it without")
print("negotiating a format.")

## 8 · Where it breaks — conformance is not accuracy

A contract check is cheap to pass and easy to over-read. Watch what else satisfies it.

In [ ]:
# An empty map. Every required key present, every type correct.
hollow = {"architecture_map": {
    "entry_points": [], "sinks": [], "flows": [], "boundaries": [],
    "reachable": [], "hotspots": [], "caveats": [],
}}
print(f"hollow map, conformance problems: {len(check(hollow, contract))}")
print(f"real map,   conformance problems: {len(check(recon, contract))}")
print()
print("Both conform. One of them found nothing at all.")
print()
print("Conformance is a statement about the serialiser: it is close to free by")
print("construction, and an empty result scores perfectly. Accuracy is the")
print("expensive part and the contract cannot measure it. Any pipeline that")
print("reports '100% schema-valid' as a quality metric is reporting this number.")
print()
print(f"what the contract can tell you : shape is right ({len(check(recon, contract))} problems)")
print(f"what only the map can tell you : {len(recon['architecture_map']['reachable'])} "
      f"reachable entry->sink pairs, {len(recon['architecture_map']['boundaries'])} "
      f"boundary crossings")
assert not check(hollow, contract), "the hollow map conforms - that is the point"

## 9 · The control — route by description, and refuse a tie

An agent picks a skill by reading descriptions. That makes the description a piece of security-relevant configuration: route wrong and you run the wrong procedure with the wrong tools.

In [ ]:
# Four skills from this repository, by description alone.
CATALOGUE = {
 "appsec-repo-recon": {"description":
   "Build the structural and historical map of a codebase before any security "
   "analysis. Use at the start of an application security review, when asked to "
   "find entry points, sinks, trust boundaries or attack surface."},
 "appsec-threat-model": {"description":
   "Turn an architecture map into a ranked, testable threat model and an audit "
   "plan. Use after repository reconnaissance, when asked what could go wrong."},
 "appsec-vuln-audit": {"description":
   "Audit code for vulnerabilities against a threat model, then deduplicate, "
   "verify in context, and filter to what is actually reachable. Use when asked "
   "to review code for security bugs or check whether a finding is a false positive."},
 "detection-triage": {"description":
   "Triage security alerts with the context needed to reach a defensible verdict. "
   "Use when working an alert queue or deciding whether an alert is a true positive."},
}

for task in ["map the attack surface of this repo before we review it",
             "what could go wrong with this architecture",
             "is this alert a false positive"]:
    pick, scores, margin = route(task, CATALOGUE)
    verdict = f"-> {pick}" if margin > 0 else f"-> AMBIGUOUS (tie at {scores[pick]})"
    print(f"{task[:44]:46s} {verdict}  margin={margin}")

print()
print("The third routes with margin 0. 'false positive' appears in the audit")
print("skill's description and 'alert' in the triage skill's, so both score the")
print("same and the winner is whichever sorted first alphabetically - an")
print("arbitrary answer wearing a confident face.")
print()
print("That is why route() returns the margin. A tie is a configuration bug in")
print("the descriptions, and the fix is to make them disjoint, not to let the")
print("sort decide which procedure runs.")
assert route("is this alert a false positive", CATALOGUE)[2] == 0

## What you just proved

Three components summarise with their entry points, outbound calls and the resources they touch. The architecture map lists two HTTP entry points, the data flows between units, and three sinks. Two trust-boundary crossings are identified, both from `src/web` into `src/data`, and both database and filesystem sinks are reachable from an entry point. Adding one handler introduces a new entry point and extends the flow graph.

## Your turn

Draw the trust-boundary edges for one service you own. The interesting output is not the diagram — it is the count of sinks reachable from an untrusted entry point, which is the number Phase 3 will spend its budget on.

---

**Next → [B1.3 · Threat modelling from the architecture map](https://spbreed.github.io/cyber-commons/lessons/B1.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B1.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B1.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*